In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import torch
import matplotlib.pyplot as plt


Sentiment Analysis

In [ ]:
classifier = pipeline("sentiment-analysis")

result = classifier("I love learning NLP!")
result2=classifier("i hate NLP")
print(result)
print(result2)

Load Tokenizer and Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"].select(range(2000)),  # small sample
    eval_dataset=encoded_dataset["test"].select(range(500)),
)

 Tokenization Example

In [ ]:
text = "I love AI"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # Initialize the tokenizer
tokens = tokenizer(text)

print(tokens)

Decode Tokens

In [ ]:
print(tokenizer.decode(tokens['input_ids']))

Load Dataset (IMDB Sentiment)

In [ ]:
dataset = load_dataset("imdb")


Preprocess / Tokenize Dataset

In [ ]:
def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

encoded_dataset = dataset.map(preprocess, batched=True)

Set Format for PyTorch

In [ ]:
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

Training

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
)

Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"].select(range(2000)),  # small sample
    eval_dataset=encoded_dataset["test"].select(range(500)),
)


Train Model

In [ ]:
train_output = trainer.train

Evaluate Model

In [ ]:
trainer.evaluate()

Models have max input length means BERT has 512 tokens only if we give the 1000 tokens last part will be removed because we are doing handling the long text or chunking

**1.do truncation / handle long text :**
here it takes only first 512 tokens
Cuts off the rest.

**2. Chunking**
Splits text into multiple smaller parts

Input:
1000 tokens

Output:
Chunk 1 → 200 tokens  
Chunk 2 → 200 tokens  
Chunk 3 → 200 tokens  
Chunk 4 → 200 tokens  
Chunk 5 → 200 tokens


Handle Long Text

In [ ]:
long_text = "Very long text..." * 1000

# tokens = tokenizer(long_text, truncation=True, max_length=512)


Handle Long Textor chunking

Total words in long_text ÷ 200 ≈ 11

In [ ]:
def chunk_text(text, max_len=200):
    words = text.split()
    for i in range(0, len(words), max_len):
        yield " ".join(words[i:i+max_len])

chunks = list(chunk_text(long_text))
print(len(chunks))


Plot Training vs Validation Loss

In [ ]:
train_loss = [0.9, 0.6, 0.4]
val_loss = [1.0, 0.7, 0.5]

plt.plot(train_loss, label="Train Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()